## StatefulSets

Ein StatefulSet in Kubernetes ist dafür gedacht, Anwendungen zu betreiben, die sich ihren Zustand merken müssen – zum Beispiel Datenbanken oder Systeme mit persistentem Speicher.

Im Gegensatz zu einem normalen Deployment behandelt ein StatefulSet seine Pods nicht als austauschbare Einheiten. Stattdessen bekommt jeder Pod eine feste Identität, die über seine gesamte Lebensdauer erhalten bleibt.

Das bedeutet konkret:

* Jeder Pod hat einen stabilen Namen (z. B. `app-0`, `app-1`, `app-2`)
* Diese Identität bleibt auch bestehen, wenn der Pod neu gestartet oder auf einen anderen Node verschoben wird
* Pods werden in einer definierten Reihenfolge gestartet, gestoppt und skaliert
* Jeder Pod kann seinen eigenen persistenten Speicher (Volume) haben

Das ist besonders wichtig für Anwendungen wie Datenbanken, bei denen ein Pod immer wieder auf “seine” Daten zugreifen muss.

---

Beispiel basierend auf [StatefulSet](https://kubernetes.io/docs/concepts/workloads/controllers/statefulset/).


In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: PersistentVolume
metadata:
  name: www-web-0
spec:
  capacity:
    storage: 1Gi
  accessModes:
  - ReadWriteOnce
  storageClassName: my-storage-class
  persistentVolumeReclaimPolicy: Retain
  hostPath:
    path: /tmp/k8s/www-web-0
---
apiVersion: v1
kind: PersistentVolume
metadata:
  name: www-web-1
spec:
  capacity:
    storage: 1Gi
  accessModes:
  - ReadWriteOnce
  storageClassName: my-storage-class
  persistentVolumeReclaimPolicy: Retain
  hostPath:
    path: /tmp/k8s/www-web-1
---
apiVersion: v1
kind: PersistentVolume
metadata:
  name: www-web-2
spec:
  capacity:
    storage: 1Gi
  accessModes:
  - ReadWriteOnce
  storageClassName: my-storage-class
  persistentVolumeReclaimPolicy: Retain
  hostPath:
    path: /tmp/k8s/www-web-2
---
apiVersion: v1
kind: Service
metadata:
  name: nginx
  labels:
    app: nginx
spec:
  ports:
  - port: 80
    name: web
  clusterIP: None
  selector:
    app: nginx
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: web
spec:
  selector:
    matchLabels:
      app: nginx
  serviceName: nginx
  replicas: 3
  minReadySeconds: 10
  template:
    metadata:
      labels:
        app: nginx
    spec:
      terminationGracePeriodSeconds: 10
      containers:
      - name: nginx
        image: registry.k8s.io/nginx-slim:0.24
        ports:
        - containerPort: 80
          name: web
        volumeMounts:
        - name: www
          mountPath: /usr/share/nginx/html
  volumeClaimTemplates:
  - metadata:
      name: www
    spec:
      accessModes:
      - ReadWriteOnce
      storageClassName: my-storage-class
      resources:
        requests:
          storage: 1Gi
EOF

In [ ]:
%%bash
kubectl get statefulsets,pods -o wide

Ein StatefulSet verhält sich bewusst anders als ein Deployment. Die Pods werden **deterministisch benannt** und behalten ihre Identität über den gesamten Lebenszyklus hinweg.
Das hat mehrere praktische Konsequenzen:

* Jeder Pod hat eine **persistente Identität** (`web-0` bleibt `web-0`)
* Die Pods werden **geordnet erstellt und beendet** (0 → 1 → 2)
* DNS-Namen sind stabil, typischerweise:
  `web-0.<service-name>`, `web-1.<service-name>`, usw.

Gerade für Microservices oder zustandsbehaftete Anwendungen ist das relevant. Ein Container kann beispielsweise über seinen Hostnamen (`web-1`) direkt seine eigene **Ordinalnummer extrahieren** und daraus ableiten:

* wie viele Instanzen vermutlich existieren (z.B. höchste bekannte Nummer)
* welche Rolle er übernehmen soll (Leader = `web-0`, Follower = Rest)
* wie er sich in ein Cluster einordnet

Das wird oft in verteilten Systemen genutzt (z.B. bei Datenbanken, Message Queues oder Leader-Election-Mechanismen), wo **stabile Identität wichtiger ist als reine Skalierung**.

Kurz gesagt:
Deployment = austauschbare, anonyme Pods
StatefulSet = identifizierbare, geordnete Pods mit stabiler Identität


---
### Aufräumen

In [ ]:
%%bash
kubectl delete statefulset web 
kubectl delete service nginx
kubectl delete pv www-web-0 www-web-1 www-web-2
kubectl delete all,pvc -l app=nginx
kubectl delete pv -l app=nginx